# Coherent Power Signal Detection

Streamlined wideband notebook for frontend correction, chunked subsection processing, and merged signal detection using only the coherent-power method.

## Pipeline Shape

1. Load a wideband spectrogram from PGM, tensor snapshot, or SigMF.
2. Apply a global frontend correction across the full band.
3. Split frequency into overlapping subsections.
4. Score each subsection with the coherent-power detector.
5. Merge subsection scores back into one wideband mask and inspect the result.

In [1]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'coherant_power_signal_detection_helpers.py').exists():
    NOTEBOOK_DIR = Path('/home/sat3737/holoscan_demo_workspace/Dinov3-RF-Signal-Detection')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import coherant_power_signal_detection_helpers as cp_helpers
cp_helpers = importlib.reload(cp_helpers)

CoherentPowerConfig = cp_helpers.CoherentPowerConfig
adapt_chunk_config_for_input_record = cp_helpers.adapt_chunk_config_for_input_record
load_input_record = cp_helpers.load_input_record
plot_chunk_plan = cp_helpers.plot_chunk_plan
plot_frontend_overview = cp_helpers.plot_frontend_overview
plot_merged_debug = cp_helpers.plot_merged_debug
plot_merged_detection = cp_helpers.plot_merged_detection
plot_subsection_debug = cp_helpers.plot_subsection_debug
run_coherent_power_pipeline = cp_helpers.run_coherent_power_pipeline

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['image.cmap'] = 'viridis'

In [2]:
PROJECT_DIR = NOTEBOOK_DIR
DEFAULT_TENSOR_DIR = Path('/tmp/usrp_spectrograms/tensors')

def latest_tensor_snapshot(channel: int | None = 0) -> Path | None:
    if not DEFAULT_TENSOR_DIR.exists():
        return None
    pattern = f'spectrogram_tensor_ch{channel}_*.npy' if channel is not None else 'spectrogram_tensor_ch*.npy'
    candidates = sorted(DEFAULT_TENSOR_DIR.glob(pattern))
    return candidates[-1] if candidates else None

DEFAULT_INPUT_PATH = latest_tensor_snapshot(channel=0) or Path('/tmp/usrp_spectrograms/spectrogram_ch0_f3_1776026453226_250x512.pgm')

INPUT_PATH = DEFAULT_INPUT_PATH
INPUT_KIND = 'auto'
SIGMF_CAPTURE_INDEX = 0
SIGMF_CHANNEL = 0
SIGMF_WINDOW_START_S = 0.0
SIGMF_WINDOW_DURATION_S = 1.0
FFT_SIZE = 20480
NOVERLAP = 15360

COHERENT_CFG = CoherentPowerConfig(
    chunk_bandwidth_hz=25e6,
    chunk_overlap_hz=6.25e6,
    uncalibrated_chunk_fraction=0.40,
    uncalibrated_overlap_fraction=0.20,
    ignore_sideband_percent=0.0,
    ignore_sideband_hz=7.0e6,
    frontend_row_q=25.0,
    frontend_reference_q=75.0,
    frontend_smooth_sigma=12.0,
    frontend_max_boost_db=12.0,
    coherence_weight=0.55,
    power_weight=0.45,
    coherence_power_support_q=0.82,
    coherence_power_q=0.92,
    min_component_size=6,
)

print({
    'input_path': str(INPUT_PATH),
    'input_kind': INPUT_KIND,
    'tensor_dir_exists': DEFAULT_TENSOR_DIR.exists(),
    'initial_subsection_bandwidth_mhz': COHERENT_CFG.chunk_bandwidth_hz / 1e6,
    'initial_subsection_overlap_mhz': COHERENT_CFG.chunk_overlap_hz / 1e6,
    'coherence_weight': COHERENT_CFG.coherence_weight,
    'power_weight': COHERENT_CFG.power_weight,
})

{'input_path': '/tmp/usrp_spectrograms/tensors/spectrogram_tensor_ch0_f5_1776039339288_625x20480.npy', 'input_kind': 'auto', 'tensor_dir_exists': True, 'initial_subsection_bandwidth_mhz': 25.0, 'initial_subsection_overlap_mhz': 6.25, 'coherence_weight': 0.55, 'power_weight': 0.45}


In [3]:
input_record = load_input_record(
    input_path=INPUT_PATH,
    input_kind=INPUT_KIND,
    fft_size=FFT_SIZE,
    noverlap=NOVERLAP,
    sigmf_capture_index=SIGMF_CAPTURE_INDEX,
    sigmf_channel=SIGMF_CHANNEL,
    sigmf_window_start_s=SIGMF_WINDOW_START_S,
    sigmf_window_duration_s=SIGMF_WINDOW_DURATION_S,
    tensor_target_height=None,
    tensor_target_width=None,
)

print({
    'input_kind': input_record['input_kind'],
    'shape': input_record['sxx_db'].shape,
    'display_shape': input_record.get('display_sxx_db', input_record['sxx_db']).shape,
    'raw_tensor_shape': input_record.get('raw_tensor_shape'),
    'resized_tensor_shape': input_record.get('resized_tensor_shape'),
    'frequency_axis_calibrated': input_record.get('frequency_axis_calibrated'),
    'sample_rate_hz': input_record['sample_rate_hz'],
    'center_frequency_hz': input_record['center_frequency_hz'],
})

{'input_kind': 'tensor_npy', 'shape': (20480, 625), 'display_shape': (625, 20480), 'raw_tensor_shape': (625, 20480), 'resized_tensor_shape': (625, 20480), 'frequency_axis_calibrated': False, 'sample_rate_hz': None, 'center_frequency_hz': None}


In [ ]:
ACTIVE_CFG = adapt_chunk_config_for_input_record(
    input_record,
    COHERENT_CFG,
    target_chunk_rows=1024,
    target_overlap_rows=256,
)

freq_axis = np.asarray(input_record['freq_axis_hz'], dtype=np.float32)
freq_bin_hz = float(np.median(np.abs(np.diff(freq_axis)))) if freq_axis.size > 1 else None
estimated_chunk_rows = None if freq_bin_hz in (None, 0.0) else int(round(ACTIVE_CFG.chunk_bandwidth_hz / freq_bin_hz))
estimated_overlap_rows = None if freq_bin_hz in (None, 0.0) else int(round(ACTIVE_CFG.chunk_overlap_hz / freq_bin_hz))

pipeline_result = run_coherent_power_pipeline(input_record, ACTIVE_CFG)
ignore_info = pipeline_result['ignore_sideband']

print({
    'active_subsection_bandwidth_mhz': round(ACTIVE_CFG.chunk_bandwidth_hz / 1e6, 3),
    'active_subsection_overlap_mhz': round(ACTIVE_CFG.chunk_overlap_hz / 1e6, 3),
    'estimated_subsection_rows': estimated_chunk_rows,
    'estimated_overlap_rows': estimated_overlap_rows,
    'subsection_count': len(pipeline_result['chunk_results']),
    'grouped_box_count': len(pipeline_result['merged_boxes']),
    'merged_threshold': round(pipeline_result['merged_threshold'], 4),
    'effective_ignore_sideband_hz_per_edge': pipeline_result['effective_ignore_sideband_hz'],
    'ignore_applied_hz_per_edge': round(float(ignore_info['applied_hz']) / 1e6, 3),
    'ignore_bins_per_edge': int(ignore_info['applied_bins']),
    'total_runtime_ms': round(pipeline_result['total_runtime_ms'], 2),
})

chunk_timing_rows = [
    {
        'subsection_index': chunk['chunk_index'],
        'rows': (chunk['row_start'], chunk['row_stop']),
        'freq_mhz': (round(chunk['freq_start_hz'] / 1e6, 3), round(chunk['freq_stop_hz'] / 1e6, 3)),
        'support_threshold': round(float(chunk['support_threshold']), 4),
        'score_threshold': round(float(chunk['score_threshold']), 4),
        **{key: round(value, 2) for key, value in chunk['timing_ms'].items()},
    }
    for chunk in pipeline_result['chunk_results']
]
chunk_timing_rows[:5]

In [ ]:
raw_sxx_db = np.asarray(input_record['sxx_db'], dtype=np.float32)
corrected_sxx_db = np.asarray(pipeline_result['corrected_sxx_db'], dtype=np.float32)
raw_row_floor = np.percentile(raw_sxx_db, ACTIVE_CFG.frontend_row_q, axis=1).astype(np.float32)
corrected_row_floor = np.percentile(corrected_sxx_db, ACTIVE_CFG.frontend_row_q, axis=1).astype(np.float32)
boost_db = np.asarray(pipeline_result['frontend']['boost_db'], dtype=np.float32)
valid_row_mask = np.asarray(pipeline_result['frontend']['valid_row_mask'], dtype=bool)

print('Frontend correction verification:')
print({
    'raw_row_floor_span_db': float(np.ptp(raw_row_floor[valid_row_mask])),
    'corrected_row_floor_span_db': float(np.ptp(corrected_row_floor[valid_row_mask])),
    'raw_row_floor_std_db': float(np.std(raw_row_floor[valid_row_mask])),
    'corrected_row_floor_std_db': float(np.std(corrected_row_floor[valid_row_mask])),
    'mean_boost_db': float(np.mean(boost_db[valid_row_mask])),
    'max_boost_db': float(np.max(boost_db[valid_row_mask])),
})

fig, _ = plot_frontend_overview(pipeline_result)
plt.show()

In [ ]:
fig, _ = plot_chunk_plan(pipeline_result)
plt.show()

fig, _ = plot_merged_detection(pipeline_result)
plt.show()

In [ ]:
fig, _ = plot_merged_debug(pipeline_result)
plt.show()

In [ ]:
SUBSECTION_DEBUG_INDEX = min(13, max(0, len(pipeline_result['chunk_results']) - 1))
chunk = next(
    candidate
    for candidate in pipeline_result['chunk_results']
    if int(candidate['chunk_index']) == int(SUBSECTION_DEBUG_INDEX)
)

print({
    'subsection_index': int(SUBSECTION_DEBUG_INDEX),
    'support_threshold': float(chunk['support_threshold']),
    'score_threshold': float(chunk['score_threshold']),
    'mask_fraction': float(np.mean(chunk['mask_px'])),
    'coherence_mean': float(np.mean(chunk['coherence_px'][chunk['valid_score_mask']])),
    'power_mean': float(np.mean(chunk['power_px'][chunk['valid_score_mask']])),
})

fig, _ = plot_subsection_debug(pipeline_result, SUBSECTION_DEBUG_INDEX)
plt.show()

subsection_grouping = cp_helpers.group_signal_mask_regions(
    np.asarray(chunk['mask_px'], dtype=bool),
    score_map=np.asarray(chunk['score_px'], dtype=np.float32),
    valid_row_mask=np.any(np.asarray(chunk['valid_score_mask'], dtype=bool), axis=1),
    bridge_freq_px=ACTIVE_CFG.grouping_bridge_freq_px,
    bridge_time_px=ACTIVE_CFG.grouping_bridge_time_px,
    min_component_size=max(int(ACTIVE_CFG.grouping_min_component_size), int(ACTIVE_CFG.min_component_size)),
    min_freq_span_px=ACTIVE_CFG.grouping_min_freq_span_px,
    min_time_span_px=ACTIVE_CFG.grouping_min_time_span_px,
    min_density=ACTIVE_CFG.grouping_min_density,
)
subsection_boxes = list(subsection_grouping['boxes'])

print({
    'subsection_box_count': len(subsection_boxes),
    'subsection_seed_threshold': float(subsection_grouping['peak_score_floor']),
    'grouped_mask_fraction': float(np.mean(np.asarray(subsection_grouping['grouped_mask'], dtype=bool))),
})

raw_chunk = np.asarray(
    pipeline_result['input_record']['sxx_db'][int(chunk['row_start']):int(chunk['row_stop']), :],
    dtype=np.float32,
)
display_transposed = bool(
    pipeline_result['input_record'].get(
        'display_transposed',
        cp_helpers.input_kind_requires_display_transpose(pipeline_result['input_record'].get('input_kind')),
    )
)
raw_vmin, raw_vmax = cp_helpers._display_db_window(raw_chunk)
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
cp_helpers._show_debug_overlay(
    ax,
    raw_chunk,
    np.asarray(subsection_grouping['grouped_mask'], dtype=bool),
    f'Subsection {SUBSECTION_DEBUG_INDEX} grouped mask + bounding boxes ({len(subsection_boxes)} boxes)',
    display_transposed,
    raw_vmin,
    raw_vmax,
    boxes=subsection_boxes,
)
plt.show()